In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Convert the categorical data to nummbers
dataset["sex"] = pd.factorize(dataset["sex"])[0]
dataset["region"] = pd.factorize(dataset["region"])[0]
dataset["smoker"] = pd.factorize(dataset["smoker"])[0]

In [ ]:
# Use 80% of the data as the training dataset and 20% of the data as the testing
# dataset without sklearn

# Shuffle the dataset before splitting
dataset = dataset.sample(frac=1)

ratio = 0.8
total_rows = dataset.shape[0]
train_size = int(total_rows*ratio)
 
# Split data into test and train
train_dataset = dataset[0:train_size]
test_dataset = dataset[train_size:]
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

In [ ]:
# Preprocessing layer to normalize features
normalizer = layers.Normalization()
normalizer.adapt(np.array(train_dataset))

# Create the model
model = keras.Sequential([
    normalizer,
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1) # Output layer for regression
])

# Compile the model
model.compile(
    optimizer=tf.optimizers.Adam(learning_rate=0.05),
    loss='mae',
    metrics=['mae', 'mse'] # Using these metrics for the testing
)

model.summary()

In [ ]:
# Train the model using the dataset and labels over 100 epochs
history = model.fit(
    train_dataset,
    train_labels,
    epochs=100
)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
